# **LSTM Model Development and Evaluation Script**

---

## Modeling Pipeline

- **Baseline Model Development:** A standard model was trained using default/reasonable hyperparameters to establish a performance benchmark.

- **Hyperparameter Optimization:** Each baseline model was optimized using two complementary techniques:
  - ***`Manual Tuning`***: Efficient, adaptive search with pruning for rapid convergence to high-performing configurations.
  - ***` Manual Random Search`***: Sklearn-compatible baseline optimization for fair comparison and reproducibility.

- **Cross-Validation and Robustness Assessment:** Each model variant was evaluated using ***`TimeSeriesSplit`*** to preserve temporal order and prevent look-ahead bias. Metrics were aggregated across folds to assess stability.

- **Overfitting Analysis:** A detailed comparison between cross-validation metrics and test set results was conducted. Additional metrics, including ***`RMSE ratio`*** and ***`R² gap`***, were computed to quantify overfitting and assess model generalization. ***`Directional accuracy`*** and ***`financial metrics`*** (Sharpe Ratio, Max Drawdown) were also calculated for trading-relevant evaluation.

---

## Persisted Artifacts

To ensure reproducibility, transparency, and extendability, the following artifacts have been saved for **each model**:

- **Optimized Model Performance:** Individual CSV files capturing the performance of each model variant:
    - ***LSTM (Baseline)***
    - ***LSTM (Manual Tuning)***
    - ***LSTM (Manual RandomizedSearch)***

- **Best Variation Performance:** A CSV file containing only the metrics of the best-performing variation per model.

- **Summary of Model Performance:** A consolidated, extendable CSV file (`AllModel_OverallPerformance.csv`) including:
    - Cross-validation results (`CV MSE`, `CV MAE`, `CV RMSE`, `CV R²`, `CV MAPE`)
    - Test set results (`Test MSE`, `Test MAE`, `Test RMSE`, `Test R²`, `Test MAPE`)
    - Financial metrics (`Sharpe Ratio`, `Sortino Ratio`, `Max Drawdown`, `Directional Accuracy`)
    - Overfitting metrics (`R² gap`, `RMSE ratio`)
    - Overfitting status and model generalization label

- **Overfitting DataFrame:** An extendable CSV (`AllModel_OverfittingAnalysis.csv`) capturing overfitting analysis metrics across all models and variations.

- **Best Model per Algorithm:** The serialized best-performing variant of each algorithm for ensemble consideration or deployment.

- **Model Comparison:** A summary notebook or script that loads `AllModel_OverallPerformance.csv` and generates publication-ready comparison visualizations.

Together, these artifacts provide a complete, reproducible record of the modeling process, facilitating model tracking, comparison, selection, and deployment.

In [ ]:
""" Configure the utilities module path for imports """
import sys
import os
from pathlib import Path

# get project root as parent of current working directory
PROJECT_ROOT = Path(os.getcwd()).parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# artifacts root
DATA_ROOT = PROJECT_ROOT / "data"
FEATURE_ROOT = PROJECT_ROOT / "artifacts" / "FeatureSelection"
FIGURE_ROOT = PROJECT_ROOT / "visualizations" / "ModelEvaluation"
MODEL_ROOT = PROJECT_ROOT / "artifacts" / "Models"
PERFORMANCE_ROOT = PROJECT_ROOT / "artifacts" / "ModelPerformance"
MODEL_CHECKPOINT = MODEL_ROOT / "Checkpoints"

In [ ]:
# records and calculations
import pandas as pd
import numpy as np

# vsiualizations
import matplotlib as plt
import seaborn as sns

# StandardScaler for scaling dataset
from sklearn.preprocessing import StandardScaler

# DL model setup
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber

import gc

# avoid minor warnings
import warnings
warnings.filterwarnings('ignore')
tf.get_logger().setLevel("ERROR")

# import utilities
from src.utilities import Evaluator, DataHandler, ModelPersister

# Load Dataset and Artifact

In [ ]:
df, x, y = DataHandler.load_dataset(DATA_ROOT / "gold_price_engineered.csv", target_col="target")
artifacts = DataHandler.load_artifacts(FEATURE_ROOT, cv_method='tscv')

In [ ]:
# check dataset loading
df.head()

In [ ]:
# check input features
x.head()

In [ ]:
# check target feature
y.head()

In [ ]:
# load train/test split data
x_train, x_test, y_train, y_test = artifacts['x_train'], artifacts['x_test'], artifacts['y_train'], artifacts['y_test']
cv = artifacts['cv']

# **Configuration Setup**

In [ ]:
# random seeds for numpy and tensorflow
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# core LSTM hyperparameters
SEQ_LEN = 20
LSTM_UNITS = [128, 64]
DROPOUT_RATE = 0.25
LEARNING_RATE = 1e-3

In [ ]:
# training schedule and callback controls
EPOCHS = 200
BATCH_SIZE = 32
PATIENCE_ES = 25
PATIENCE_LR = 10
LR_FACTOR = 0.5
MIN_LR = 1e-6

In [ ]:
# feature and target scaling
scaler_x = StandardScaler()
scaler_y = StandardScaler()

x_train_scaler = scaler_x.fit_transform(x_train)
x_test_scaler = scaler_x.transform(x_test)

y_train_scaler = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaler = scaler_y.transform(y_test.values.reshape(-1, 1))

# **Build Baseline Model**

## Create Sequence and Define Callbacks

In [ ]:
def create_sequences(x, y, seq_len):
    x_seq, y_seq = [], []
    
    for i in range(seq_len, len(x)):
        x_seq.append(x[i - seq_len:i])
        y_seq.append(y[i])

    return np.array(x_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)

In [ ]:
x_train_seq, y_train_seq = create_sequences(x_train_scaler, y_train_scaler, SEQ_LEN)
x_test_seq, y_test_seq = create_sequences(x_test_scaler, y_test_scaler, SEQ_LEN)

In [ ]:
print(f"x_train_seq shape : {x_train_seq.shape}")
print(f"x_test_seq  shape : {x_test_seq.shape}")
print(f"y_train_seq shape : {y_train_seq.shape}")
print(f"y_test_seq  shape : {y_test_seq.shape}")

In [ ]:
# build lstm model
baseline_model = Sequential(name="LSTM_Baseline")

for idx, unit in enumerate(LSTM_UNITS):
    return_seq = idx < len(LSTM_UNITS) - 1

    baseline_model.add(LSTM(
        units=unit,
        return_sequences=return_seq,
        kernel_initializer="glorot_uniform",
        recurrent_initializer="orthogonal",
        name=f"lstm_{idx+1}",
    ))

    baseline_model.add(Dropout(
        DROPOUT_RATE,
        name=f"dropout_{idx+1}"
    ))

    baseline_model.add(BatchNormalization(
        name=f"batchnorm_{idx+1}"
    ))

baseline_model.add(Dense(1, activation="linear", name="output"))
loss = Huber(delta=1.0)

In [ ]:
baseline_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE, clipnorm=1.0),
    loss=loss,
    metrics=["mae"]
)

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE_ES,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=LR_FACTOR,
        patience=PATIENCE_LR,
        min_lr=MIN_LR,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=str(MODEL_CHECKPOINT / "LSTM_baseline_best.weights.h5"),
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=0
    )
]

## Train Baseline Model

In [ ]:
history = baseline_model.fit(
    x_train_seq, y_train_seq,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=0,
    shuffle=False
)

In [ ]:
baseline_model.summary()

In [ ]:
# extract best validation performance
best_val_loss = min(history.history["val_loss"])
best_epoch = np.argmin(history.history["val_loss"])

print("Best Val Loss:", best_val_loss)
print("Best Epoch:", best_epoch)

## Apply Model to Make Prediction

In [ ]:
# make prediction on train sequence
train_pred_scaled = baseline_model.predict(x_train_seq).flatten()
y_train_pred_baseline = scaler_y.inverse_transform(train_pred_scaled.reshape(-1, 1)).flatten()

# make prediction on test sequence
test_pred_scaled = baseline_model.predict(x_test_seq).flatten()
y_test_pred_baseline = scaler_y.inverse_transform(test_pred_scaled.reshape(-1, 1)).flatten()

## Evaluating the Model Performance

In [ ]:
# evaluation of train metrics
train_metrics_baseline = Evaluator.calculate_metrics(y_train_seq, y_train_pred_baseline)

# evaluation of test metrics
test_metrics_baseline = Evaluator.calculate_metrics(y_test_seq, y_test_pred_baseline)

In [ ]:
# calculate directional time-series specific accuracy
train_acc_baseline = Evaluator.directional_accuracy(y_train_seq, y_train_pred_baseline)
test_acc_baseline = Evaluator.directional_accuracy(y_test_seq, y_test_pred_baseline)

In [ ]:
# calcluate financial metrics of test data for baseline model
fin_baseline = Evaluator.financial_metrics('LSTM (Baseline)', y_test_seq, y_test_pred_baseline)

display(fin_baseline)

In [ ]:
# baseline model performance
baseline_perf = Evaluator.performance_table(train_metrics_baseline + [train_acc_baseline], test_metrics_baseline + [test_acc_baseline])

print("LSTM - Baseline Modeling Performance")
display(baseline_perf)

# **Structured Manual Tuning**

## Tuning Utility & Model Builder

In [ ]:
# training utility
def run_experiment(
        build_model,
        x_train, y_train,
        x_test, y_test,
        scaler_y, experiment_name="ManualTuning",
        seq_len=SEQ_LEN, epochs=EPOCHS,
        batch_size=BATCH_SIZE, val_split=0.2,
        patience_es=PATIENCE_ES, patience_lr=PATIENCE_LR,
        lr_factor=LR_FACTOR, min_lr=MIN_LR,
        architecture=None, dropout=None,
        learning_rate=None, verbose=0
):
    
    # clear previous session to free memory
    tf.keras.backend.clear_session()
    gc.collect()

    # initialize model with input shape
    model = build_model(input_shape=(x_train.shape[1], x_train.shape[2]))

    # stops training when validation loss stops improving
    es = EarlyStopping(
        monitor="val_loss", patience=patience_es,
        restore_best_weights=True, verbose=0
    )

    # reduce learning rate when validation loss plateaus
    rlr = ReduceLROnPlateau(
        monitor="val_loss", factor=lr_factor,
        patience=patience_lr, min_lr=min_lr,  verbose=0
    )

    # saves best model weights based on validation loss
    mc = ModelCheckpoint(
        filepath=str(MODEL_CHECKPOINT / f"LSTM_{experiment_name}_best.weights.h5"),
        monitor="val_loss", save_best_only=True,
        save_weights_only=True, verbose=0
    )

    # train the model
    hist = model.fit(
        x_train, y_train,
        validation_split=val_split,
        epochs=epochs, batch_size=batch_size,
        callbacks=[es, rlr, mc],
        verbose=verbose, shuffle=False,
    )

    # make prediction on scaled input
    train_pred_scaled = model.predict(x_train).flatten()
    test_pred_scaled = model.predict(x_test).flatten()

    # inverse transform predictions back to original scale
    y_train_pred = scaler_y.inverse_transform(train_pred_scaled.reshape(-1, 1)).flatten()
    y_test_pred = scaler_y.inverse_transform(test_pred_scaled.reshape(-1, 1)).flatten()

    # compute regression metrics
    train_metrics = Evaluator.calculate_metrics(y_train, y_train_pred)
    test_metrics = Evaluator.calculate_metrics(y_test, y_test_pred)

    # compute directioal accuracy
    train_acc = Evaluator.directional_accuracy(y_train, y_train_pred)
    test_acc = Evaluator.directional_accuracy(y_test, y_test_pred)

    # extract best validation performance
    best_val_loss = min(hist.history["val_loss"])
    best_epoch = np.argmin(hist.history["val_loss"]) + 1

    # aggregate experiment result
    result = {
        "experiment": experiment_name,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "seq_len": seq_len,
        "architecture": architecture,
        "dropout": dropout,
        "learning_rate": learning_rate,
        "batch_size": batch_size,

        # train metrics
        "train_MSE": train_metrics[0],
        "train_MAE": train_metrics[1],
        "train_RMSE": train_metrics[2],
        "train_R2": train_metrics[3],
        "train_MAPE": train_metrics[4],
        "train_Dir_Acc": train_acc,

        # test metrics
        "test_MSE": test_metrics[0],
        "test_MAE": test_metrics[1],
        "test_RMSE": test_metrics[2],
        "test_R2": test_metrics[3],
        "test_MAPE": test_metrics[4],
        "test_Dir_Acc": test_acc,

    }

    tf.keras.backend.clear_session()
    gc.collect()
    
    return result, model, hist

In [ ]:
# create LSTM model builder function for experiment
def lstm_builder(lstm_units, dropout=0.25, lr=1e-3, loss="huber", bidirectional=False):
    def builder(input_shape=(x_train.shape[1], x_train.shape[2])):
        model = Sequential(name=f"LSTM_{'_'.join(map(str, lstm_units))}")

        for idx, unit in enumerate(lstm_units):
            ret_seq = idx < len(lstm_units) - 1
            layer = LSTM(
                units=unit,
                return_sequences=ret_seq,
                kernel_initializer="glorot_uniform",
                recurrent_initializer="orthogonal",
                name=f"lstm_{idx+1}"
            )

            if bidirectional:
                layer = Bidirectional(layer, name=f"bilstm_{idx+1}")
            
            model.add(layer)
            model.add(Dropout(dropout, name=f"dropout_{idx+1}"))
            model.add(BatchNormalization(name=f"batchnorm_{idx+1}"))
        
        model.add(Dense(1, activation="linear", name="output"))
        loss = Huber(delta=1.0)

        model.compile(optimizer=Adam(learning_rate=lr, clipnorm=1.0),  loss=loss, metrics=["mae"])
        
        return model

    return builder

## Hyperparameter Tuning

### Sequence Length Tuning

In [ ]:
SEQ_EXPERIMENTS = [10, 15, 20, 25, 30]

seq_results = []

for sl in SEQ_EXPERIMENTS:
    x_train, y_train = create_sequences(x_train_scaler, y_train_scaler, seq_len=sl)
    x_test, y_test = create_sequences(x_test_scaler, y_test_scaler, seq_len=sl)

    r, _, _ = run_experiment(
        lstm_builder([128, 64]),
        x_train, y_train,
        x_test, y_test,
        scaler_y=scaler_y,
        experiment_name=f"seq{sl}",
        seq_len=sl,
        epochs=200,
        batch_size=32,
        verbose=0,
    )
    seq_results.append(r)

best_seq = min(seq_results, key=lambda x: x["test_RMSE"])
BEST_SEQ_LEN = int(best_seq["seq_len"])
TEST_RMSE = round(best_seq["test_RMSE"], 4)

print(f">>> Best sequence length: {BEST_SEQ_LEN}  (test_RMSE={TEST_RMSE})")

In [ ]:
# lock the best sequence for further tuning
x_train_seq, y_train_seq = create_sequences(x_train_scaler, y_train_scaler, BEST_SEQ_LEN)
x_test_seq, y_test_seq = create_sequences(x_test_scaler, y_test_scaler, BEST_SEQ_LEN)

### Architecture Tuning

In [ ]:
ARCHITECTURES_EXPERIMENT = {
    "arch_1x64":         [64],
    "arch_1x128":        [128],
    "arch_2x64_32":      [64, 32],
    "arch_2x128_64":     [128, 64],
    "arch_3x128_64_32":  [128, 64, 32],
}

arch_result = []

for name, unit in ARCHITECTURES_EXPERIMENT.items():
    r, _, _ = run_experiment(
        lstm_builder(unit),
        x_train_seq, y_train_seq,
        x_test_seq, y_test_seq,
        scaler_y=scaler_y,
        experiment_name=name,
        seq_len=BEST_SEQ_LEN,
        architecture=unit,
        epochs=200,
        batch_size=32,
        verbose=0
    )

    arch_result.append(r)

best_arch = min(arch_result, key=lambda x: x["test_RMSE"])
BEST_UNIT = best_arch["architecture"]
TEST_RMSE = round(best_arch["test_RMSE"], 4)

print(f">>> Best architecture: {BEST_UNIT}  (test_RMSE={TEST_RMSE})")

### Dropout Tuning

In [ ]:
DROPOUT_EXPERIMENT = [0.2, 0.25, 0.3, 0.4, 0.5]

drop_result = []

for drop in DROPOUT_EXPERIMENT:
    r, _, _ = run_experiment(
        lstm_builder(BEST_UNIT, dropout=drop),
        x_train_seq, y_train_seq,
        x_test_seq, y_test_seq,
        scaler_y=scaler_y,
        experiment_name=f"drop{int(drop*100)}",
        seq_len=BEST_SEQ_LEN,
        architecture=BEST_UNIT,
        dropout=drop,
        epochs=200,
        batch_size=32,
        verbose=0
    )

    drop_result.append(r)

best_drop = min(drop_result, key=lambda x: x["test_RMSE"])
BEST_DROPOUT = best_drop["dropout"]
TEST_RMSE = round(best_drop["test_RMSE"], 4)

print(f">>> Best dropout rate: {BEST_DROPOUT}  (test_RMSE={TEST_RMSE})")

### Learning Rate Tuning

In [ ]:
LR_EXPERIMENT =  [1e-4, 5e-4, 1e-3, 3e-3, 5e-3]

lr_result = []

for lr in LR_EXPERIMENT:
    r, _, _ = run_experiment(
        lstm_builder(BEST_UNIT, dropout=BEST_DROPOUT, lr=lr),
        x_train_seq, y_train_seq,
        x_test_seq, y_test_seq,
        scaler_y=scaler_y,
        experiment_name=f"lr{lr}",
        seq_len=BEST_SEQ_LEN,
        architecture=BEST_UNIT,
        dropout=BEST_DROPOUT,
        learning_rate=lr,
        epochs=200,
        batch_size=32,
        verbose=0
    )

    lr_result.append(r)

best_lr = min(lr_result, key=lambda x: x["test_RMSE"])

BEST_LR = best_lr["learning_rate"]
TEST_RMSE = round(best_lr["test_RMSE"], 4)

print(f">>> Best learning rate: {BEST_LR}  (test_RMSE={TEST_RMSE})")

### Batch Size Tuning

In [ ]:
BATCH_EXPERIMENT = [16, 32, 64, 128]

batch_result = []

for batch in BATCH_EXPERIMENT:
    r, _, _ = run_experiment(
        lstm_builder(BEST_UNIT, dropout=BEST_DROPOUT, lr=BEST_LR),
        x_train_seq, y_train_seq,
        x_test_seq, y_test_seq,
        scaler_y=scaler_y,
        experiment_name=f"batch{batch}",
        seq_len=BEST_SEQ_LEN,
        architecture=BEST_UNIT,
        dropout=BEST_DROPOUT,
        learning_rate=BEST_LR,
        batch_size=batch,
    )

    batch_result.append(r)

best_batch = min(batch_result, key=lambda x: x["test_RMSE"])

BEST_BATCH = best_batch["batch_size"]
TEST_RMSE = round(best_batch["test_RMSE"], 4)

print(f">>> Best batch size: {BEST_BATCH}  (test_RMSE={TEST_RMSE})")

### Loss Function Tuning

In [ ]:
loss_result = []

In [ ]:
r_mse, _, _ = run_experiment(
    lstm_builder(BEST_UNIT, dropout=BEST_DROPOUT, lr=BEST_LR, loss="mse"),
    x_train_seq, y_train_seq,
    x_test_seq, y_test_seq,
    scaler_y=scaler_y,
    experiment_name="loss-mse",
    seq_len=BEST_SEQ_LEN,
    epochs=200,
    batch_size=BEST_BATCH,
    verbose=0,
)

loss_result.append(r_mse)

In [ ]:
r_huber, _, _ = run_experiment(
    lstm_builder(BEST_UNIT, dropout=BEST_DROPOUT, lr=BEST_LR, loss="huber"),
    x_train_seq, y_train_seq,
    x_test_seq, y_test_seq,
    scaler_y=scaler_y,
    experiment_name="loss-huber",
    seq_len=BEST_SEQ_LEN,
    epochs=200,
    batch_size=BEST_BATCH,
    verbose=0
)

loss_result.append(r_huber)

In [ ]:
best_loss = min(loss_result, key=lambda x: x["test_RMSE"])

BEST_LOSS = best_loss["experiment"][5:]                # remove 'loss' suffix
TEST_RMSE = round(best_loss["test_RMSE"], 4)

print(f">>> Best loss function: {BEST_LOSS}  (test_RMSE={TEST_RMSE})")

### Bidirectional Tuning

In [ ]:
results_tracker = pd.DataFrame()

r, _, _ = run_experiment(
    lstm_builder(BEST_UNIT, dropout=BEST_DROPOUT, lr=BEST_LR, loss=BEST_LOSS, bidirectional=True),
    x_train_seq, y_train_seq,
    x_test_seq, y_test_seq,
    scaler_y=scaler_y,
    experiment_name="bidirectional",
    seq_len=BEST_SEQ_LEN,
    architecture=BEST_UNIT,
    dropout=BEST_DROPOUT,
    learning_rate=BEST_LR,
    batch_size=BEST_BATCH,
    epochs=200,
    verbose=0
)

row = pd.DataFrame([r])

results_tracker = pd.concat([results_tracker, row], ignore_index=True)

best_rmse = results_tracker[~results_tracker["experiment"].isin(["baseline",  "bidirectional"])]["test_RMSE"].min()

if r["test_RMSE"] < best_rmse:
    print(">>> Bidirectional is BETTER")
    USE_BIDIRECTIONAL = True

else:
    print(">>> Standard LSTM is better")
    USE_BIDIRECTIONAL = False

## Build and Train Manual Tuned Model

In [ ]:
# configuration setup for final model training and evaluation
MANUAL_CONFIG = {
    "seq_len": BEST_SEQ_LEN,
    "lstm_units": BEST_UNIT,
    "dropout_rate": BEST_DROPOUT,
    "learning_rate": BEST_LR,
    "batch_size": BEST_BATCH,
    "loss_fn" : BEST_LOSS,
    "bidirectional": USE_BIDIRECTIONAL
    }

In [ ]:
# create sequence for train and test set
x_train_seq, y_train_seq = create_sequences(x_train_scaler, y_train_scaler, MANUAL_CONFIG["seq_len"])
x_test_seq, y_test_seq = create_sequences(x_test_scaler, y_test_scaler, MANUAL_CONFIG["seq_len"])

In [ ]:
tf.keras.backend.clear_session()
gc.collect()

In [ ]:
# build final model with best configuration
manual_model = Sequential(name="LSTM_ManualTuning")

for idx, unit in enumerate(MANUAL_CONFIG["lstm_units"]):
    return_seq = idx < len(MANUAL_CONFIG["lstm_units"]) - 1
    
    layer = LSTM(
        units=unit,
        return_sequences=return_seq,
        kernel_initializer="glorot_uniform",
        recurrent_initializer="orthogonal",
        name=f"lstm_{idx+1}",
    )

    if MANUAL_CONFIG["bidirectional"]:
        layer = Bidirectional(layer, name=f"bilstm_{idx+1}")
    
    manual_model.add(layer)
    manual_model.add(Dropout(MANUAL_CONFIG["dropout_rate"], name=f"dropout_{idx+1}"))
    manual_model.add(BatchNormalization(name=f"batchnorm_{idx+1}"))

manual_model.add(Dense(1, activation="linear", name="output"))
loss_obj = Huber(delta=1.0) if MANUAL_CONFIG["loss_fn"] == "huber" else "mse"

In [ ]:
# compile the model
manual_model.compile(
    optimizer=Adam(learning_rate=MANUAL_CONFIG["learning_rate"], clipnorm=1.0),
    loss=loss_obj,
    metrics=["mae"]
)

In [ ]:
# model callbacks for final training
final_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=30,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=15,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=str(MODEL_CHECKPOINT / "LSTM_ManualTuning_best.weights.h5"),
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=0
    )
]

In [ ]:
#  fit the final model
history = manual_model.fit(
    x_train_seq, y_train_seq,
    validation_split=0.2,
    epochs=300,
    batch_size=MANUAL_CONFIG["batch_size"],
    callbacks=final_callbacks,
    verbose=0,
    shuffle=False
)

In [ ]:
# summary of the manual tuned model
manual_model.summary()

In [ ]:
# extract best validation performance
best_val_loss = round(min(history.history["val_loss"]), 4)
best_epoch = np.argmin(history.history["val_loss"]) + 1

print("Best Val Loss:", best_val_loss)
print("Best Epoch:", best_epoch)

## Apply Model to Make Prediction

In [ ]:
# make prediction on train sequence
train_pred_scaled = manual_model.predict(x_train_seq).flatten()
y_train_pred_manual = scaler_y.inverse_transform(train_pred_scaled.reshape(-1, 1)).flatten()

# make prediction on test sequence
test_pred_scaled = manual_model.predict(x_test_seq).flatten()
y_test_pred_manual = scaler_y.inverse_transform(test_pred_scaled.reshape(-1, 1)).flatten()

## Evaluation of Maual Tuned Model Performance

In [ ]:
# evaluation of train metrics
train_metrics_manual = Evaluator.calculate_metrics(y_train_seq, y_train_pred_manual)

# evaluation of test metrics
test_metrics_manual = Evaluator.calculate_metrics(y_test_seq, y_test_pred_manual)

In [ ]:
# calciulate directional accuracy
train_acc_manual = Evaluator.directional_accuracy(y_train_seq, y_train_pred_manual)
test_acc_manual = Evaluator.directional_accuracy(y_test_seq, y_test_pred_manual)

In [ ]:
# calculate financial metrics of test data for manual tuned model
fin_manual = Evaluator.financial_metrics('LSTM (Manual Tuning)', y_test_seq, y_test_pred_manual)

display(fin_manual)

In [ ]:
# manual tuning model performance
manual_perf = Evaluator.performance_table(train_metrics_manual + [train_acc_manual], test_metrics_manual + [test_acc_manual])

print("LSTM - Manual Tuning Performance")
display(manual_perf)

# **Manual RandomSearch Tuning**

## Random Search Sapce (Narrowed Around Manual Tuning)

In [ ]:
# randomized search configuration
N_ITER = 30
RS_EPOCHS = 200
RS_PATIENCE_ES = 25
RS_PATIENCE_LR = 10
RS_LR_FACTOR = 0.5
RS_MIN_LR = 1e-6

In [ ]:
# narrowed search space centered around manual tuning best
SEARCH_SPACE = {
    "seq_len": [25, 30, 35, 40, 45],
    "lstm_units": [
        [64],
        [128],
        [64, 32],
        [128, 64],
        [128, 64, 32]
    ],
    "dropout": [0.25, 0.3, 0.35, 0.4, 0.45],
    "learning_rate": [1e-3, 3e-3, 5e-3, 5e-5, 7e-3],
    "batch_size" : [16, 32, 64, 128],
    "loss": ["huber", "mse"],
    "bidirectional": [False, True]
}

In [ ]:
# compute total search space
total_combo = 1
for value in SEARCH_SPACE.values():
    total_combo += len(value)

In [ ]:
print(f"Search space size : {total_combo} possible combinations")
print(f"Will test         : {N_ITER} random combinations")
print(f"Epochs per trial  : {RS_EPOCHS}")
print(f"ES patience       : {RS_PATIENCE_ES}")
print(f"LR patience       : {RS_PATIENCE_LR}")

## Randomized Search Engine

In [ ]:
import random

def RandomizedSearch(
        search_space, n_iter,
        x_train_scaler, y_train_scaler,
        x_test_scaler, y_test_scaler,
        scaler_y, epochs=RS_EPOCHS,
        patience_es=RS_PATIENCE_ES, patience_lr=RS_PATIENCE_LR,
        lr_factor=RS_LR_FACTOR, min_lr=RS_MIN_LR,
        seed=SEED
):
    tf.keras.backend.clear_session()
    gc.collect()

    random.seed(seed)
    np.random.seed(seed)

    all_results = []

    for i in range(n_iter):
        trial = {
            "seq_len" : random.choice(search_space["seq_len"]),
            "lstm_units": random.choice(search_space["lstm_units"]),
            "dropout": random.choice(search_space["dropout"]),
            "learning_rate": random.choice(search_space["learning_rate"]),
            "batch_size": random.choice(search_space["batch_size"]),
            "loss": random.choice(search_space["loss"]),
            "bidirectional": random.choice(search_space["bidirectional"])
        }

        exp_name = (
            f"rs_{i+1}_seq{trial['seq_len']}"
            f"_{'x'.join(map(str, trial['lstm_units']))}"
            f"_d{int(trial['dropout']*100)}"
            f"_lr{trial['learning_rate']}"
            f"_b{trial['batch_size']}"
        )

        x_train, y_train = create_sequences(x_train_scaler, y_train_scaler, trial["seq_len"])
        x_test, y_test = create_sequences(x_test_scaler, y_test_scaler, trial["seq_len"])

        result, model, hist = run_experiment(
            build_model=lstm_builder(
                lstm_units= trial["lstm_units"],
                dropout=trial["dropout"],
                lr=trial["learning_rate"],
                loss=trial["loss"],
                bidirectional=trial["bidirectional"]
            ),

            experiment_name=exp_name,
            x_train=x_train, y_train=y_train,
            x_test=x_test, y_test=y_test,
            scaler_y=scaler_y, epochs=epochs,
            seq_len=trial["seq_len"], batch_size=trial["batch_size"],
            val_split=0.2,
            patience_es=patience_es, patience_lr=patience_lr,
            lr_factor=lr_factor, min_lr=min_lr,
            architecture=trial["lstm_units"], dropout=trial["dropout"],
            learning_rate=trial["learning_rate"],
            verbose=0
        )

        result["trial_num"] = i + 1
        result["bidirectional"] = trial["bidirectional"]
        result["loss_fn"] = trial["loss"]

        all_results.append(result)

        # clean up
        del model, hist, x_train, y_train, x_test, y_test
        tf.keras.backend.clear_session()
        gc.collect()
    
    return all_results

In [ ]:
# run randomized search

rs_result = RandomizedSearch(
    search_space=SEARCH_SPACE, n_iter=N_ITER,
    x_train_scaler=x_train_scaler, y_train_scaler=y_train_scaler,
    x_test_scaler=x_test_scaler, y_test_scaler=y_test_scaler,
    scaler_y=scaler_y, epochs=RS_EPOCHS,
    patience_es=RS_PATIENCE_ES, patience_lr=RS_PATIENCE_LR,
    lr_factor=RS_LR_FACTOR, min_lr=RS_MIN_LR,
    seed=SEED
)

## Analyze Search Results

In [ ]:
# convert search result into dataframe
rs_df = pd.DataFrame(rs_result)
rs_df = rs_df.sort_values(by="test_RMSE", ascending=True).reset_index(drop=True)

In [ ]:
# display search results
print("Top 10 trials ranked by test_RMSE")
display(rs_df[[
    "trial_num", "experiment", "best_epoch", "best_val_loss",
    "seq_len", "architecture", "dropout", "learning_rate",
    "batch_size", "loss_fn", "bidirectional",
    "test_RMSE", "test_R2", "test_Dir_Acc",
]].head(10))

## Build and Train Manual RandomizedSearch Model

In [ ]:
# first row of the sorted dataframe is the best trial
best_idx = rs_df.iloc[0]

# configuration setup for final model training and evaluation
RANDOM_CONFIG = {
    "seq_len": int(best_idx["seq_len"]),
    "lstm_units": best_idx["architecture"],
    "dropout_rate": best_idx["dropout"],
    "learning_rate": best_idx["learning_rate"],
    "batch_size": int(best_idx["batch_size"]),
    "loss_fn" : best_idx["loss_fn"],
    "bidirectional": best_idx["bidirectional"]
}

In [ ]:
# create sequence for train and test set
x_train_seq, y_train_seq = create_sequences(x_train_scaler, y_train_scaler, RANDOM_CONFIG["seq_len"])
x_test_seq, y_test_seq = create_sequences(x_test_scaler, y_test_scaler, RANDOM_CONFIG["seq_len"])

In [ ]:
# clear previous session
tf.keras.backend.clear_session()
gc.collect()

In [ ]:
# build final model with best random search configuration
random_model = Sequential(name="LSTM_ManualRandomSearch")

for idx, unit in enumerate(RANDOM_CONFIG["lstm_units"]):
    return_seq = idx < len(RANDOM_CONFIG["lstm_units"]) - 1
    
    layer = LSTM(
        units=unit,
        return_sequences=return_seq,
        kernel_initializer="glorot_uniform",
        recurrent_initializer="orthogonal",
        name=f"lstm_rs_{idx+1}",
    )

    if RANDOM_CONFIG["bidirectional"]:
        layer = Bidirectional(layer, name=f"bilstm_rs_{idx+1}")

    random_model.add(layer)
    random_model.add(Dropout(RANDOM_CONFIG["dropout_rate"], name=f"dropout_rs_{idx+1}"))
    random_model.add(BatchNormalization(name=f"batchnorm_rs_{idx+1}"))

random_model.add(Dense(1, activation="linear", name="rs_output"))

loss_obj = Huber(delta=1.0) if RANDOM_CONFIG["loss_fn"] == "huber" else "mse"

In [ ]:
# compile the model
random_model.compile(
    optimizer=Adam(learning_rate=RANDOM_CONFIG["learning_rate"], clipnorm=1.0),
    loss=loss_obj,
    metrics=["mae"]
)

In [ ]:
# model callbacks for final training
rs_callback = [
    EarlyStopping(
        monitor="val_loss",
        patience=30,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=15,
        min_lr=1e-6,
        verbose=1
    ),

    ModelCheckpoint(
        filepath=str(MODEL_CHECKPOINT / "LSTM_RandomSearch_best.weights.h5"),
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=0
    )
]

In [ ]:
# fit the final model from random search
rs_history = random_model.fit(
    x_train_seq, y_train_seq,
    validation_split=0.2,
    epochs=300,
    batch_size=RANDOM_CONFIG["batch_size"],
    callbacks=rs_callback,
    verbose=0,
    shuffle=False
)

In [ ]:
# summary of the randomized search model
random_model.summary()

In [ ]:
# extract best validation performance
best_val_loss = round(min(rs_history.history["val_loss"]), 4)
best_epoch = np.argmin(rs_history.history["val_loss"]) + 1

print("Best Val Loss:", best_val_loss)
print("Best Epoch:", best_epoch)

## Apply Model to Make Prediction

In [ ]:
# make prediction on train sequence
train_pred_scaled = random_model.predict(x_train_seq).flatten()
y_train_pred_random = scaler_y.inverse_transform(train_pred_scaled.reshape(-1, 1)).flatten()

# make prediction on test sequence
test_pred_scaled = random_model.predict(x_test_seq).flatten()
y_test_pred_random = scaler_y.inverse_transform(test_pred_scaled.reshape(-1, 1)).flatten()

## Evaluation of RandomizedSearch Model Performance

In [ ]:
# evaluation of train metrics
train_metrics_random = Evaluator.calculate_metrics(y_train_seq, y_train_pred_random)

# evaluation of test metrics
test_metrics_random = Evaluator.calculate_metrics(y_test_seq, y_test_pred_random)

In [ ]:
# calciulate directional accuracy
train_acc_random = Evaluator.directional_accuracy(y_train_seq, y_train_pred_random)
test_acc_random = Evaluator.directional_accuracy(y_test_seq, y_test_pred_random)

In [ ]:
# calculate financial metrics of test data for random search model
fin_random = Evaluator.financial_metrics('LSTM (Randomized Search)', y_test_seq, y_test_pred_random)

display(fin_random)

In [ ]:
# random search model performance
random_perf = Evaluator.performance_table(train_metrics_random + [train_acc_random], test_metrics_random + [test_acc_random])
    
print("LSTM - Randomized Search Performance")
display(random_perf)

# Cross Valiation (All Models)

## LSTM CV Setup

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def lstm_cv_evaluate(
    build_model_fn,
    x_train, y_train,
    cv, seq_len,
    epochs=200, batch_size=32,
    patience_es=25, patience_lr=10,
    lr_factor=0.5, min_lr=1e-6,
    verbose=0,
):
    
    if isinstance(x_train, pd.DataFrame):
        x_arr = x_train.values.astype(np.float32)
    
    else:
        x_arr = x_train.astype(np.float32)

    if isinstance(y_train, pd.Series):
        y_arr = y_train.values.astype(np.float32)
    
    else:
        y_arr = y_train.astype(np.float32)

    metrics = []

    for fold_idx, (train_idx, val_idx) in enumerate(cv.split(x_arr, y_arr)):

        tf.keras.backend.clear_session()
        gc.collect()

        # per-fold split
        x_fold_train = x_arr[train_idx]
        y_fold_train = y_arr[train_idx]
        x_fold_val   = x_arr[val_idx]
        y_fold_val   = y_arr[val_idx]

        # per-fold scaling
        fold_scaler_x = StandardScaler()
        fold_scaler_y = StandardScaler()

        x_train_scaler = fold_scaler_x.fit_transform(x_fold_train)
        x_val_scaler = fold_scaler_x.transform(x_fold_val)

        y_train_scaler = fold_scaler_y.fit_transform(y_fold_train.reshape(-1, 1))
        y_val_scaler = fold_scaler_y.transform(y_fold_val.reshape(-1, 1))

        # per-fold sequences
        x_train_seq, y_train_seq = create_sequences(x_train_scaler, y_train_scaler, seq_len)
        x_val_seq, y_val_seq = create_sequences(x_val_scaler, y_val_scaler, seq_len)

        # build fresh model
        model = build_model_fn(input_shape=(x_train_seq.shape[1], x_train_seq.shape[2]))

        # callbacks
        es = EarlyStopping(
            monitor="val_loss",
            patience=patience_es,
            restore_best_weights=True,
            verbose=0,
        )
        rlr = ReduceLROnPlateau(
            monitor="val_loss",
            factor=lr_factor,
            patience=patience_lr,
            min_lr=min_lr,
            verbose=0,
        )

        # train
        model.fit(
            x_train_seq, y_train_seq,
            validation_data=(x_val_seq, y_val_seq),
            epochs=epochs, batch_size=batch_size,
            callbacks=[es, rlr],
            verbose=verbose,
            shuffle=False,
        )

        # predict and inverse-transform
        val_pred_scaler = model.predict(x_val_seq, verbose=0).flatten()
        val_pred = fold_scaler_y.inverse_transform(val_pred_scaler.reshape(-1, 1)).flatten()
        val_true = y_fold_val[seq_len:]  # align after sequence trim

        # evaluate
        fold_mse  = mean_squared_error(val_true, val_pred)
        fold_mae  = mean_absolute_error(val_true, val_pred)
        fold_rmse = np.sqrt(fold_mse)
        fold_r2   = r2_score(val_true, val_pred)
        fold_mape = Evaluator.safe_mape(val_true, val_pred)
        fold_dir  = Evaluator.directional_accuracy(val_true, val_pred)

        metrics.append({
            "MSE": fold_mse,
            "MAE": fold_mae,
            "RMSE": fold_rmse,
            "R2": fold_r2,
            "MAPE": fold_mape,
            "Dir_Acc": fold_dir,
        })

        print(f"  Fold {fold_idx + 1}: RMSE={fold_rmse:.4f}, "
              f"R2={fold_r2:.4f}, Dir_Acc={fold_dir:.2f}%")

        tf.keras.backend.clear_session()
        gc.collect()

    # aggregate
    metrics_df = pd.DataFrame(metrics)

    return {
        "CV MSE": metrics_df["MSE"].mean(),
        "CV MAE": metrics_df["MAE"].mean(),
        "CV RMSE": metrics_df["RMSE"].mean(),
        "CV R2": metrics_df["R2"].mean(),
        "CV MAPE": metrics_df["MAPE"].mean(),
        "CV Directional Accuracy (%)": metrics_df["Dir_Acc"].mean(),
    }

## Model Builder Configuration

In [ ]:
# baseline mdoel
def build_baseline(input_shape):
    model = Sequential(name="CV_LSTM_Baseline")
    for idx, unit in enumerate(LSTM_UNITS):
        return_seq = idx < len(LSTM_UNITS) - 1

        layer = LSTM(
            units=unit,
            return_sequences=return_seq,
            kernel_initializer="glorot_uniform",
            recurrent_initializer="orthogonal",
            name=f"cv_lstm_{idx+1}",
        )

        model.add(layer)
        model.add(Dropout(DROPOUT_RATE, name=f"cv_dropout_{idx+1}"))
        model.add(BatchNormalization(name=f"cv_batchnorm_{idx+1}"))
    
    model.add(Dense(1, activation="linear", name="cv_output"))
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE, clipnorm=1.0), loss=Huber(delta=1.0), metrics=["mae"])
    
    return model

In [ ]:
# build manual model
def build_manual(input_shape):
    model = Sequential(name="CV_LSTM_ManualTuning")

    for idx, unit in enumerate(MANUAL_CONFIG["lstm_units"]):
        return_seq = idx < len(MANUAL_CONFIG["lstm_units"]) - 1
        
        layer = LSTM(
            units=unit,
            return_sequences=return_seq,
            kernel_initializer="glorot_uniform",
            recurrent_initializer="orthogonal",
            name=f"cv_manual_lstm_{idx+1}",
        )

        if MANUAL_CONFIG["bidirectional"]:
            layer = Bidirectional(layer, name=f"bilstm_{idx+1}")
        
        model.add(layer)
        model.add(Dropout(MANUAL_CONFIG["dropout_rate"], name=f"cv_manual_dropout_{idx+1}"))
        model.add(BatchNormalization(name=f"cv_manual_batchnorm_{idx+1}"))

    model.add(Dense(1, activation="linear", name="cv_manual_output"))
    loss_obj = Huber(delta=1.0) if MANUAL_CONFIG["loss_fn"] == "huber" else "mse"

    model.compile(optimizer=Adam(learning_rate=MANUAL_CONFIG["learning_rate"], clipnorm=1.0), loss=loss_obj, metrics=["mae"])
    
    return model

In [ ]:
# build random search model
def build_random(input_shape):
    model = Sequential(name="CV_LSTM_RandomSearch")

    for idx, unit in enumerate(RANDOM_CONFIG["lstm_units"]):
        return_seq = idx < len(RANDOM_CONFIG["lstm_units"]) - 1
        
        layer = LSTM(
            units=unit,
            return_sequences=return_seq,
            kernel_initializer="glorot_uniform",
            recurrent_initializer="orthogonal",
            name=f"cv_random_lstm_{idx+1}",
        )

        if RANDOM_CONFIG["bidirectional"]:
            layer = Bidirectional(layer, name=f"cv_bilstm_random_{idx+1}")

        model.add(layer)
        model.add(Dropout(RANDOM_CONFIG["dropout_rate"], name=f"cv_random_dropout_{idx+1}"))
        model.add(BatchNormalization(name=f"cv_random_batchnorm_{idx+1}"))

    model.add(Dense(1, activation="linear", name="cv_random_output"))
    loss_obj = Huber(delta=1.0) if RANDOM_CONFIG["loss_fn"] == "huber" else "mse"
    model.compile(optimizer=Adam(learning_rate=RANDOM_CONFIG["learning_rate"], clipnorm=1.0), loss=loss_obj, metrics=["mae"])

    return  model

## Cross Validation Result

In [ ]:
# CV baseline
cv_baseline = lstm_cv_evaluate(
    build_model_fn=build_baseline,
    x_train=artifacts["x_train"], y_train=artifacts["y_train"],
    cv=cv, seq_len=20,
    epochs=200, batch_size=32,
    verbose=0)

In [ ]:
# cv manual tuning
cv_manual = lstm_cv_evaluate(
    build_model_fn=build_manual,
    x_train=artifacts["x_train"], y_train=artifacts["y_train"],
    cv=cv, seq_len=MANUAL_CONFIG["seq_len"],
    epochs=200, batch_size=MANUAL_CONFIG["batch_size"],
    verbose=0)

In [ ]:
# CV randomsearch
cv_random = lstm_cv_evaluate(
    build_model_fn=build_random,
    x_train=artifacts["x_train"], y_train=artifacts["y_train"],
    cv=cv, seq_len=RANDOM_CONFIG["seq_len"],
    epochs=200, batch_size=RANDOM_CONFIG["batch_size"],
    verbose=0
)